# A simple LLM compression using SVD

In [19]:
import math
from typing import Optional, Tuple

import torch
import torch.utils.checkpoint
from torch import nn

from transformers.activations import ACT2FN
from transformers.utils import logging
from transformers import LlamaConfig, LlamaForCausalLM, LlamaTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = LlamaTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

print(model)

Some parameters are on the meta device because they were offloaded to the cpu and disk.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): 

## List of layers to be compressed (from official paper implementation)

In [20]:
original_layers = []
original_layers.append(model.model.layers[0].self_attn.q_proj)
original_layers.append(model.model.layers[0].self_attn.k_proj)
original_layers.append(model.model.layers[0].self_attn.v_proj)
original_layers.append(model.model.layers[0].self_attn.o_proj)
original_layers.append(model.model.layers[0].mlp.gate_proj)
original_layers.append(model.model.layers[0].mlp.up_proj)
original_layers.append(model.model.layers[0].mlp.down_proj)

compress_layers = []
compress_layers.append(model.model.layers[0].self_attn.q_proj)
compress_layers.append(model.model.layers[0].self_attn.k_proj)
compress_layers.append(model.model.layers[0].self_attn.v_proj)
compress_layers.append(model.model.layers[0].self_attn.o_proj)
compress_layers.append(model.model.layers[0].mlp.gate_proj)
compress_layers.append(model.model.layers[0].mlp.up_proj)
compress_layers.append(model.model.layers[0].mlp.down_proj)

## Function to test compression performance. A low relative error is ideal.

In [21]:
def test_svd_compression(original_layer, compressed_model, batch_size=10):
    # Create random input batch (adjust input size to layer)
    input_dim = original_layer.in_features
    x = torch.randn(batch_size, input_dim)

    # Ensure x dtype matches compressed_model's first parameter dtype (weights)
    target_dtype = next(compressed_model.parameters()).dtype
    x = x.to(target_dtype)

    with torch.no_grad():
        y_orig = original_layer(x)
        y_compressed = compressed_model(x)

    diff = (y_orig - y_compressed).float()  # for stable error computation in float32
    rel_error = diff.norm() / y_orig.float().norm()

    print(f"Relative error after compression: {rel_error.item():.6f}")


## A baseline compression using vanilla SVD

In [22]:
def layer_SVD_baseline(layer: nn.Linear, old_layer: nn.Linear, ratio=0.9):
    device = layer.weight.device
    dtype = layer.weight.dtype

    weight = layer.weight.data.float().to(device)  # Cast to float32 for SVD stability

    # Compute full SVD of weight matrix
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)

    # Determine rank to keep based on energy ratio
    total_energy = (S**2).sum()
    energy_cutoff = ratio * total_energy

    running_energy = 0.0
    rank = 0
    for s in S:
        running_energy += s**2
        rank += 1
        if running_energy >= energy_cutoff:
            break

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    input_dim = weight.shape[1]
    output_dim = weight.shape[0]

    # Compose new weights for compressed layers
    new1_weight = torch.diag(S_k) @ Vh_k
    new2_weight = U_k

    # Create two linear layers to replace the original
    new_1 = nn.Linear(input_dim, rank, bias=False).to(device=device, dtype=dtype)
    new_2 = nn.Linear(rank, output_dim, bias=False).to(device=device, dtype=dtype)

    # Copy weights back, cast to original dtype
    new_1.weight.data.copy_(new1_weight.to(dtype))
    new_2.weight.data.copy_(new2_weight.to(dtype))

    return nn.Sequential(new_1, new_2)


## Compress the layers and replace them in the model

In [23]:
for i in range(len(compress_layers)):
    compressed = layer_SVD_baseline(compress_layers[i], original_layers[i])
    test_svd_compression(original_layers[i], compressed, batch_size=10)
    # Replace the original layer
    if i == 0:
        model.model.layers[0].self_attn.q_proj = compressed
    elif i == 1:
        model.model.layers[0].self_attn.k_proj = compressed
    elif i == 2:
        model.model.layers[0].self_attn.v_proj = compressed
    elif i == 3:
        model.model.layers[0].self_attn.o_proj = compressed
    elif i == 4:
        model.model.layers[0].mlp.gate_proj = compressed
    elif i == 5:
        model.model.layers[0].mlp.up_proj = compressed
    elif i == 6:
        model.model.layers[0].mlp.down_proj = compressed

Relative error after compression: 0.331856
Relative error after compression: 0.323180
Relative error after compression: 0.310690
Relative error after compression: 0.308059
Relative error after compression: 0.324051
Relative error after compression: 0.314120
Relative error after compression: 0.317753


## View model with replaced layers

In [24]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Sequential(
            (0): Linear(in_features=2048, out_features=84, bias=False)
            (1): Linear(in_features=84, out_features=2048, bias=False)
          )
          (k_proj): Sequential(
            (0): Linear(in_features=2048, out_features=22, bias=False)
            (1): Linear(in_features=22, out_features=256, bias=False)
          )
          (v_proj): Sequential(
            (0): Linear(in_features=2048, out_features=189, bias=False)
            (1): Linear(in_features=189, out_features=256, bias=False)
          )
          (o_proj): Sequential(
            (0): Linear(in_features=2048, out_features=583, bias=False)
            (1): Linear(in_features=583, out_features=2048, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): Sequential(
       

## Key Take-Aways
The two matrices that approximate each layer have fewer parameters than the original layer. For example, the original q_proj had 2048 x 2048 = 4,194,304 parameters. The compressed q_proj (consisting of two smaller matrices) has (2048 x 84) + (84 x 2048) = 344,064 parameters.